# Image Captioning with CNN-LSTM and Attention

This notebook implements an automatic image captioning system using:
- **Encoder**: ResNet-50 CNN for feature extraction
- **Decoder**: LSTM with Bahdanau Attention for caption generation
- **Dataset**: Flickr8k

**GPU Support**: This notebook is configured to automatically use your GPU if CUDA is available.

## Cell 1: Import Libraries

In [ ]:
import os
import re
import random
import urllib.request
import zipfile
from collections import Counter

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tqdm.notebook import tqdm

# Download NLTK data if needed
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count: {torch.cuda.device_count()}")

## Cell 2: Configuration

In [ ]:
class Config:
    """Hyperparameters and paths"""
    # Data paths
    data_dir = "data/Flickr8k"
    images_dir = os.path.join(data_dir, "Images")
    captions_file = os.path.join(data_dir, "captions.txt")

    # Training split ratios
    train_ratio = 0.8
    val_ratio = 0.1  # test = 0.1

    # Vocabulary settings
    min_freq = 5
    max_caption_len = 40

    # Model architecture
    embed_dim = 256
    encoder_dim = 512
    decoder_dim = 512
    attention_dim = 256

    # Training hyperparameters
    batch_size = 64  # Increase if you have more GPU memory
    num_workers = 4  # Adjust based on your CPU cores
    lr = 1e-4
    num_epochs = 10
    grad_clip = 5.0
    
    # Device - GPU will be used automatically if available
    device = (
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )

    # Special tokens
    pad_token = "<pad>"
    start_token = "<start>"
    end_token = "<end>"
    unk_token = "<unk>"

cfg = Config()
print(f"\n{'='*60}")
print(f"Configuration")
print(f"{'='*60}")
print(f"Device: {cfg.device}")
print(f"Batch size: {cfg.batch_size}")
print(f"Epochs: {cfg.num_epochs}")
print(f"Learning rate: {cfg.lr}")
print(f"{'='*60}")

## Cell 3: Download Dataset

In [ ]:
def download_flickr8k():
    """Download and extract Flickr8k dataset"""
    print("Downloading Flickr8k dataset...")
    
    # Create directories
    os.makedirs('data/Flickr8k', exist_ok=True)
    
    # Download URLs
    dataset_url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip"
    captions_url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"
    
    # Download images
    if not os.path.exists("Flickr8k_Dataset.zip"):
        print("Downloading images...")
        urllib.request.urlretrieve(dataset_url, "Flickr8k_Dataset.zip")
    
    # Download captions
    if not os.path.exists("Flickr8k_text.zip"):
        print("Downloading captions...")
        urllib.request.urlretrieve(captions_url, "Flickr8k_text.zip")
    
    # Extract
    if not os.path.exists(cfg.images_dir):
        print("Extracting images...")
        with zipfile.ZipFile("Flickr8k_Dataset.zip", 'r') as zip_ref:
            zip_ref.extractall("data/Flickr8k")
    
    if not os.path.exists(cfg.captions_file):
        print("Extracting captions...")
        with zipfile.ZipFile("Flickr8k_text.zip", 'r') as zip_ref:
            zip_ref.extractall("data/Flickr8k")
        
        # Rename captions file
        if os.path.exists("data/Flickr8k/Flickr8k.token.txt"):
            os.rename("data/Flickr8k/Flickr8k.token.txt", cfg.captions_file)
    
    print("✓ Dataset ready!")

# Download dataset if not already present
if not os.path.exists(cfg.captions_file):
    download_flickr8k()
else:
    print("✓ Dataset already downloaded.")

## Cell 4: Utility Functions

In [ ]:
def clean_caption(caption: str) -> str:
    """Clean and normalize caption text"""
    caption = caption.lower().strip()
    caption = re.sub(r"[^a-z0-9,.!?']", " ", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption

def tokenize(caption: str):
    """Simple whitespace tokenization"""
    return caption.split()

# Test the functions
test_caption = "A DOG running through the field!"
print(f"Original: {test_caption}")
print(f"Cleaned: {clean_caption(test_caption)}")
print(f"Tokenized: {tokenize(clean_caption(test_caption))}")

## Cell 5: Vocabulary Class

In [ ]:
class Vocabulary:
    """Build and manage vocabulary"""
    def __init__(self, min_freq=5):
        self.min_freq = min_freq
        self.freqs = Counter()
        self.stoi = {}
        self.itos = []

        self.pad_token = cfg.pad_token
        self.start_token = cfg.start_token
        self.end_token = cfg.end_token
        self.unk_token = cfg.unk_token

    def build(self, all_captions):
        """Build vocabulary from captions"""
        # Count word frequencies
        for cap in all_captions:
            tokens = tokenize(clean_caption(cap))
            self.freqs.update(tokens)

        # Add special tokens first
        self.itos = [
            self.pad_token,
            self.start_token,
            self.end_token,
            self.unk_token,
        ]
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}

        # Add words that meet frequency threshold
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def numericalize(self, caption: str):
        """Convert caption to token IDs"""
        tokens = tokenize(clean_caption(caption))
        return [self.stoi.get(tok, self.stoi[self.unk_token]) for tok in tokens]

    def __len__(self):
        return len(self.itos)

print("✓ Vocabulary class defined")

## Cell 6: Data Loading Functions

In [ ]:
def load_captions(captions_file):
    """
    Load captions from file
    
    Returns:
        image2caps: dict mapping image_name to list of captions
        all_pairs: list of (image_name, caption) tuples
    """
    image2caps = {}
    all_pairs = []

    with open(captions_file, "r") as f:
        for line in f:
            if not line.strip():
                continue

            # Split on first comma
            parts = line.strip().split(",", 1)
            if len(parts) != 2:
                continue
                
            img_id, caption = parts
            img_name = img_id.strip()
            caption = caption.strip()

            # Skip header or non-image rows
            if not img_name.lower().endswith(".jpg"):
                continue

            image2caps.setdefault(img_name, []).append(caption)
            all_pairs.append((img_name, caption))

    return image2caps, all_pairs


def train_val_test_split(all_pairs, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Split data into train/val/test sets"""
    random.seed(seed)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)
    
    train_data = all_pairs[:n_train]
    val_data = all_pairs[n_train:n_train + n_val]
    test_data = all_pairs[n_train + n_val:]
    
    return train_data, val_data, test_data

print("✓ Data loading functions defined")

## Cell 7: Dataset Class

In [ ]:
class FlickrCaptionDataset(Dataset):
    """Flickr8k dataset loader"""
    def __init__(self, image_caption_pairs, images_dir, vocab: Vocabulary,
                 transform=None, max_len=40):
        self.data = image_caption_pairs
        self.images_dir = images_dir
        self.vocab = vocab
        self.transform = transform
        self.max_len = max_len

        self.start_idx = vocab.stoi[cfg.start_token]
        self.end_idx = vocab.stoi[cfg.end_token]
        self.pad_idx = vocab.stoi[cfg.pad_token]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name, caption = self.data[idx]
        img_path = os.path.join(self.images_dir, img_name)

        # Load image
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Encode caption: <start> tokens <end>
        token_ids = self.vocab.numericalize(caption)
        token_ids = token_ids[: self.max_len - 2]
        caption_ids = [self.start_idx] + token_ids + [self.end_idx]
        length = len(caption_ids)

        return image, torch.tensor(caption_ids, dtype=torch.long), length


class CaptionCollate:
    """Custom collate function for variable-length captions"""
    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        images, captions, lengths = zip(*batch)
        images = torch.stack(images, dim=0)

        # Pad captions to max length in batch
        max_len = max(lengths)
        padded_captions = torch.full(
            (len(captions), max_len),
            fill_value=self.pad_idx,
            dtype=torch.long,
        )

        for i, cap in enumerate(captions):
            end = cap.shape[0]
            padded_captions[i, :end] = cap

        lengths = torch.tensor(lengths, dtype=torch.long)

        return images, padded_captions, lengths

print("✓ Dataset classes defined")

## Cell 8: Encoder (ResNet-50 CNN)

In [ ]:
class EncoderCNN(nn.Module):
    """ResNet-50 based encoder"""
    def __init__(self, encoded_image_size=14, encoder_dim=512):
        super().__init__()
        self.enc_image_size = encoded_image_size

        # Load pretrained ResNet-50
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        
        # Remove final layers (keep conv features)
        modules = list(resnet.children())[:-2]
        self.cnn = nn.Sequential(*modules)

        # Adaptive pooling to fixed size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))

        # Project from 2048 to encoder_dim
        self.conv_project = nn.Conv2d(2048, encoder_dim, kernel_size=1, stride=1)

        self.fine_tune(False)

    def forward(self, images):
        """
        Args:
            images: (B, 3, H, W)
        Returns:
            features: (B, L, encoder_dim) where L = enc_size^2
        """
        features = self.cnn(images)              # (B, 2048, H', W')
        features = self.adaptive_pool(features)   # (B, 2048, enc_size, enc_size)
        features = self.conv_project(features)    # (B, encoder_dim, enc_size, enc_size)
        
        # Reshape to sequence
        B, D, H, W = features.size()
        features = features.permute(0, 2, 3, 1).view(B, -1, D)  # (B, L, D)
        
        return features

    def fine_tune(self, fine_tune=True):
        """Freeze/unfreeze encoder layers"""
        for p in self.cnn.parameters():
            p.requires_grad = False
        # Optionally unfreeze later layers
        for c in list(self.cnn.children())[-2:]:
            for p in c.parameters():
                p.requires_grad = fine_tune

print("✓ Encoder defined")

## Cell 9: Attention Mechanism

In [ ]:
class BahdanauAttention(nn.Module):
    """Additive attention mechanism"""
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        """
        Args:
            encoder_out: (B, L, encoder_dim)
            decoder_hidden: (B, decoder_dim)
        Returns:
            context: (B, encoder_dim)
            alpha: (B, L) attention weights
        """
        att1 = self.encoder_att(encoder_out)  # (B, L, att_dim)
        att2 = self.decoder_att(decoder_hidden).unsqueeze(1)  # (B, 1, att_dim)
        att = self.full_att(self.relu(att1 + att2)).squeeze(2)  # (B, L)
        alpha = self.softmax(att)  # (B, L)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)  # (B, encoder_dim)
        return context, alpha

print("✓ Attention mechanism defined")

## Cell 10: Decoder (LSTM with Attention)

In [ ]:
class DecoderWithAttention(nn.Module):
    """LSTM decoder with attention"""
    def __init__(self, vocab_size, embed_dim, encoder_dim, decoder_dim, attention_dim, pad_idx):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx

        self.attention = BahdanauAttention(encoder_dim, decoder_dim, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        
        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)
        
        self.lstm_cell = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)
        self.sigmoid = nn.Sigmoid()
        
        self.fc = nn.Linear(decoder_dim, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def init_hidden_state(self, encoder_out):
        """Initialize LSTM state from encoder output"""
        mean_encoder = encoder_out.mean(dim=1)  # (B, encoder_dim)
        h = self.init_h(mean_encoder)
        c = self.init_c(mean_encoder)
        return h, c

    def forward(self, encoder_out, captions, lengths):
        """
        Args:
            encoder_out: (B, L, encoder_dim)
            captions: (B, max_len)
            lengths: (B,)
        Returns:
            predictions: (B, max_len-1, vocab_size)
            captions_sorted: sorted captions
            decode_lengths: actual decode lengths
            alphas: attention weights
            sort_idx: sorting indices
        """
        batch_size = encoder_out.size(0)
        L = encoder_out.size(1)

        # Sort by length
        lengths_sorted, sort_idx = lengths.sort(dim=0, descending=True)
        encoder_out = encoder_out[sort_idx]
        captions = captions[sort_idx]

        # Embed captions
        embeddings = self.embedding(captions)  # (B, max_len, embed_dim)

        # Initialize LSTM
        h, c = self.init_hidden_state(encoder_out)

        # Decode lengths
        decode_lengths = (lengths_sorted - 1).tolist()
        max_decode_len = max(decode_lengths)

        # Storage
        predictions = torch.zeros(batch_size, max_decode_len, self.vocab_size).to(encoder_out.device)
        alphas = torch.zeros(batch_size, max_decode_len, L).to(encoder_out.device)

        # Decode step by step
        for t in range(max_decode_len):
            batch_t = sum([l > t for l in decode_lengths])
            
            # Attention
            context, alpha = self.attention(encoder_out[:batch_t], h[:batch_t])
            gate = self.sigmoid(self.f_beta(h[:batch_t]))
            context = gate * context

            # LSTM step
            lstm_input = torch.cat([embeddings[:batch_t, t, :], context], dim=1)
            h_new, c_new = self.lstm_cell(lstm_input, (h[:batch_t], c[:batch_t]))

            # Update states
            h = torch.cat([h_new, h[batch_t:]], dim=0)
            c = torch.cat([c_new, c[batch_t:]], dim=0)

            # Predict
            preds = self.fc(self.dropout(h_new))
            predictions[:batch_t, t, :] = preds
            alphas[:batch_t, t, :] = alpha

        return predictions, captions, decode_lengths, alphas, sort_idx

print("✓ Decoder defined")

## Cell 11: Create Data Loaders

In [ ]:
def create_dataloaders():
    """Create train/val/test dataloaders"""
    # Load captions
    image2caps, all_pairs = load_captions(cfg.captions_file)
    train_pairs, val_pairs, test_pairs = train_val_test_split(
        all_pairs, cfg.train_ratio, cfg.val_ratio
    )

    # Build vocabulary
    vocab = Vocabulary(min_freq=cfg.min_freq)
    vocab.build([cap for _, cap in train_pairs])
    cfg.pad_token_id = vocab.stoi[cfg.pad_token]

    print(f"Vocabulary size: {len(vocab)}")
    print(f"Training samples: {len(train_pairs)}")
    print(f"Validation samples: {len(val_pairs)}")
    print(f"Test samples: {len(test_pairs)}")

    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Datasets
    train_dataset = FlickrCaptionDataset(
        train_pairs, cfg.images_dir, vocab,
        transform=train_transform, max_len=cfg.max_caption_len
    )
    val_dataset = FlickrCaptionDataset(
        val_pairs, cfg.images_dir, vocab,
        transform=val_transform, max_len=cfg.max_caption_len
    )
    test_dataset = FlickrCaptionDataset(
        test_pairs, cfg.images_dir, vocab,
        transform=val_transform, max_len=cfg.max_caption_len
    )

    collate = CaptionCollate(pad_idx=cfg.pad_token_id)

    # Dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, collate_fn=collate, pin_memory=True if cfg.device == 'cuda' else False
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, collate_fn=collate, pin_memory=True if cfg.device == 'cuda' else False
    )
    test_loader = DataLoader(
        test_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, collate_fn=collate, pin_memory=True if cfg.device == 'cuda' else False
    )

    return train_loader, val_loader, test_loader, vocab, image2caps, test_pairs

# Create the dataloaders
train_loader, val_loader, test_loader, vocab, image2caps, test_pairs = create_dataloaders()
print("\n✓ Data loaders created")

## Cell 12: Training Functions

In [ ]:
def train_one_epoch(encoder, decoder, criterion, optimizer, train_loader, epoch):
    """Train for one epoch"""
    encoder.train()
    decoder.train()

    total_loss = 0.0

    for images, captions, lengths in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        images = images.to(cfg.device)
        captions = captions.to(cfg.device)
        lengths = lengths.to(cfg.device)

        optimizer.zero_grad()

        encoder_out = encoder(images)
        scores, caps_sorted, decode_lengths, alphas, sort_idx = decoder(
            encoder_out, captions, lengths
        )

        # Targets: next word after each position
        targets = caps_sorted[:, 1:]

        # Pack predictions and targets
        scores_packed = []
        targets_packed = []
        for i, l in enumerate(decode_lengths):
            scores_packed.append(scores[i, :l, :])
            targets_packed.append(targets[i, :l])

        scores_packed = torch.cat(scores_packed, dim=0)
        targets_packed = torch.cat(targets_packed, dim=0)

        loss = criterion(scores_packed, targets_packed)

        # Attention regularization (doubly stochastic)
        alphas_reg = 1.0 * ((1.0 - alphas.sum(dim=1)) ** 2).mean()
        loss = loss + alphas_reg

        loss.backward()
        nn.utils.clip_grad_norm_(decoder.parameters(), cfg.grad_clip)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


@torch.no_grad()
def validate(encoder, decoder, criterion, val_loader, epoch):
    """Validate the model"""
    encoder.eval()
    decoder.eval()

    total_loss = 0.0

    for images, captions, lengths in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
        images = images.to(cfg.device)
        captions = captions.to(cfg.device)
        lengths = lengths.to(cfg.device)

        encoder_out = encoder(images)
        scores, caps_sorted, decode_lengths, alphas, sort_idx = decoder(
            encoder_out, captions, lengths
        )

        targets = caps_sorted[:, 1:]

        scores_packed = []
        targets_packed = []
        for i, l in enumerate(decode_lengths):
            scores_packed.append(scores[i, :l, :])
            targets_packed.append(targets[i, :l])

        scores_packed = torch.cat(scores_packed, dim=0)
        targets_packed = torch.cat(targets_packed, dim=0)

        loss = criterion(scores_packed, targets_packed)
        alphas_reg = 1.0 * ((1.0 - alphas.sum(dim=1)) ** 2).mean()
        loss = loss + alphas_reg

        total_loss += loss.item()

    return total_loss / len(val_loader)

print("✓ Training functions defined")

## Cell 13: Initialize Model and Optimizer

In [ ]:
# Build models
encoder = EncoderCNN(encoder_dim=cfg.encoder_dim).to(cfg.device)
decoder = DecoderWithAttention(
    vocab_size=len(vocab),
    embed_dim=cfg.embed_dim,
    encoder_dim=cfg.encoder_dim,
    decoder_dim=cfg.decoder_dim,
    attention_dim=cfg.attention_dim,
    pad_idx=vocab.stoi[cfg.pad_token],
).to(cfg.device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi[cfg.pad_token])
params = list(decoder.parameters()) + list(
    filter(lambda p: p.requires_grad, encoder.parameters())
)
optimizer = optim.Adam(params, lr=cfg.lr)

print(f"\n{'='*60}")
print(f"Model initialized on {cfg.device}")
print(f"Total trainable parameters: {sum(p.numel() for p in params if p.requires_grad):,}")
print(f"{'='*60}")

## Cell 14: Main Training Loop

In [ ]:
print("="*60)
print("Starting Training")
print("="*60)

best_val_loss = float("inf")
train_losses = []
val_losses = []

# Training loop
for epoch in range(1, cfg.num_epochs + 1):
    train_loss = train_one_epoch(
        encoder, decoder, criterion, optimizer, train_loader, epoch
    )
    val_loss = validate(encoder, decoder, criterion, val_loader, epoch)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"\nEpoch {epoch}/{cfg.num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        
        checkpoint = {
            "encoder": encoder.state_dict(),
            "decoder": decoder.state_dict(),
            "vocab": {
                "itos": vocab.itos,
                "stoi": vocab.stoi,
            },
            "config": {
                "encoder_dim": cfg.encoder_dim,
                "decoder_dim": cfg.decoder_dim,
                "embed_dim": cfg.embed_dim,
                "attention_dim": cfg.attention_dim,
            },
        }
        
        torch.save(checkpoint, "best_model.pth")
        print("  ✓ Saved best model")

print("\n" + "="*60)
print("Training Complete!")
print("="*60)

## Cell 15: Plot Training Curves

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss vs Epoch')
plt.legend()
plt.grid(True)
plt.show()

print(f"Best validation loss: {best_val_loss:.4f}")

## Cell 16: Caption Generation Function

In [ ]:
@torch.no_grad()
def generate_caption(encoder, decoder, image_tensor, vocab, max_len=20):
    """Generate caption for a single image"""
    encoder.eval()
    decoder.eval()

    image_tensor = image_tensor.to(cfg.device)
    encoder_out = encoder(image_tensor)

    h, c = decoder.init_hidden_state(encoder_out)
    start_idx = vocab.stoi[cfg.start_token]
    end_idx = vocab.stoi[cfg.end_token]

    word_idx = start_idx
    caption_idxs = [start_idx]

    for _ in range(max_len):
        word = torch.tensor([word_idx], dtype=torch.long).to(cfg.device)
        embeddings = decoder.embedding(word)

        context, alpha = decoder.attention(encoder_out, h)
        gate = decoder.sigmoid(decoder.f_beta(h))
        context = gate * context

        lstm_input = torch.cat([embeddings, context], dim=1)
        h, c = decoder.lstm_cell(lstm_input, (h, c))
        
        preds = decoder.fc(h)
        _, next_word = preds.max(dim=1)

        word_idx = next_word.item()
        caption_idxs.append(word_idx)
        
        if word_idx == end_idx:
            break

    # Convert to words
    words = []
    for idx in caption_idxs:
        tok = vocab.itos[idx]
        if tok in {cfg.start_token, cfg.end_token, cfg.pad_token}:
            continue
        words.append(tok)

    return " ".join(words)

print("✓ Caption generation function defined")

## Cell 17: BLEU Evaluation Function

In [ ]:
@torch.no_grad()
def evaluate_bleu_on_test(encoder, decoder, vocab, image2caps, test_pairs):
    """Compute BLEU-4 on test set"""
    encoder.eval()
    decoder.eval()

    # Get unique test images
    test_images = sorted(set(img_name for img_name, _ in test_pairs))

    references = []
    hypotheses = []

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    print(f"\nEvaluating BLEU on {len(test_images)} test images...")

    for img_name in tqdm(test_images, desc="BLEU eval"):
        img_path = os.path.join(cfg.images_dir, img_name)

        # Load image
        pil_img = Image.open(img_path).convert("RGB")
        img_tensor = transform(pil_img).unsqueeze(0)

        # Generate caption
        hyp_str = generate_caption(encoder, decoder, img_tensor, vocab, max_len=20)
        hyp_tokens = hyp_str.split()
        hypotheses.append(hyp_tokens)

        # Get reference captions
        ref_caps = image2caps[img_name]
        ref_tokens = []
        for cap in ref_caps:
            toks = tokenize(clean_caption(cap))
            ref_tokens.append(toks)
        references.append(ref_tokens)

    # BLEU-4 with smoothing
    smoothie = SmoothingFunction().method1
    bleu4 = corpus_bleu(references, hypotheses, smoothing_function=smoothie)
    
    print(f"\n{'='*60}")
    print(f"Test BLEU-4 Score: {bleu4:.4f}")
    print('='*60)
    
    return bleu4, references, hypotheses

print("✓ BLEU evaluation function defined")

## Cell 18: Run BLEU Evaluation

In [ ]:
# Evaluate on test set
bleu4_score, references, hypotheses = evaluate_bleu_on_test(
    encoder, decoder, vocab, image2caps, test_pairs
)

## Cell 19: Visualize Sample Predictions

In [ ]:
def visualize_predictions(encoder, decoder, vocab, test_pairs, num_samples=5):
    """Visualize sample predictions"""
    encoder.eval()
    decoder.eval()
    
    # Get random test images
    test_images = list(set(img_name for img_name, _ in test_pairs))
    sample_images = random.sample(test_images, min(num_samples, len(test_images)))
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = [axes]
    
    for idx, img_name in enumerate(sample_images):
        img_path = os.path.join(cfg.images_dir, img_name)
        
        # Load and display image
        pil_img = Image.open(img_path).convert("RGB")
        img_tensor = transform(pil_img).unsqueeze(0)
        
        # Generate caption
        generated_caption = generate_caption(encoder, decoder, img_tensor, vocab, max_len=20)
        
        # Get ground truth captions
        gt_captions = image2caps[img_name]
        
        # Display
        axes[idx].imshow(pil_img)
        axes[idx].axis('off')
        axes[idx].set_title(
            f"Generated: {generated_caption}\n" +
            f"Ground Truth 1: {gt_captions[0][:80]}...",
            fontsize=10, wrap=True
        )
    
    plt.tight_layout()
    plt.show()

# Visualize 5 random predictions
visualize_predictions(encoder, decoder, vocab, test_pairs, num_samples=5)

## Cell 20: Generate Caption for Custom Image

In [ ]:
def caption_custom_image(image_path, encoder, decoder, vocab):
    """Generate caption for a custom image"""
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    # Load image
    pil_img = Image.open(image_path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0)
    
    # Generate caption
    caption = generate_caption(encoder, decoder, img_tensor, vocab, max_len=20)
    
    # Display
    plt.figure(figsize=(10, 6))
    plt.imshow(pil_img)
    plt.axis('off')
    plt.title(f"Generated Caption: {caption}", fontsize=12, pad=10)
    plt.show()
    
    return caption

# Example: Caption a specific image from test set
test_image = random.choice(list(set(img_name for img_name, _ in test_pairs)))
test_image_path = os.path.join(cfg.images_dir, test_image)
generated_caption = caption_custom_image(test_image_path, encoder, decoder, vocab)
print(f"\nGenerated: {generated_caption}")
print(f"\nGround Truth Captions:")
for i, cap in enumerate(image2caps[test_image], 1):
    print(f"{i}. {cap}")

## Cell 21: Save and Load Model Functions

In [ ]:
def save_model(encoder, decoder, vocab, filepath="image_captioning_model.pth"):
    """Save the trained model"""
    checkpoint = {
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "vocab": {
            "itos": vocab.itos,
            "stoi": vocab.stoi,
        },
        "config": {
            "encoder_dim": cfg.encoder_dim,
            "decoder_dim": cfg.decoder_dim,
            "embed_dim": cfg.embed_dim,
            "attention_dim": cfg.attention_dim,
        },
    }
    torch.save(checkpoint, filepath)
    print(f"✓ Model saved to {filepath}")


def load_model(filepath="best_model.pth", device="cuda"):
    """Load a trained model"""
    checkpoint = torch.load(filepath, map_location=device)
    
    # Reconstruct vocabulary
    vocab = Vocabulary()
    vocab.itos = checkpoint["vocab"]["itos"]
    vocab.stoi = checkpoint["vocab"]["stoi"]
    
    # Reconstruct models
    config = checkpoint["config"]
    encoder = EncoderCNN(encoder_dim=config["encoder_dim"]).to(device)
    decoder = DecoderWithAttention(
        vocab_size=len(vocab),
        embed_dim=config["embed_dim"],
        encoder_dim=config["encoder_dim"],
        decoder_dim=config["decoder_dim"],
        attention_dim=config["attention_dim"],
        pad_idx=vocab.stoi["<pad>"],
    ).to(device)
    
    # Load weights
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
    
    encoder.eval()
    decoder.eval()
    
    print(f"✓ Model loaded from {filepath}")
    return encoder, decoder, vocab

# Example: Save the current model
save_model(encoder, decoder, vocab, "final_model.pth")

# Example: Load the best model
# encoder, decoder, vocab = load_model("best_model.pth", device=cfg.device)

## Cell 22: Summary and Next Steps

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
print(f"Device used: {cfg.device}")
print(f"Epochs trained: {cfg.num_epochs}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Test BLEU-4 score: {bleu4_score:.4f}")
print(f"Vocabulary size: {len(vocab)}")
print(f"Model saved to: best_model.pth")
print("="*60)
print("\nNext steps:")
print("1. Try generating captions for your own images")
print("2. Fine-tune with more epochs or different hyperparameters")
print("3. Experiment with beam search for better caption quality")
print("4. Visualize attention weights for interpretability")
print("5. Try on larger datasets like Flickr30k or MS COCO")
print("="*60)

## Optional: Load Previously Saved Model

If you want to load a previously trained model without retraining, uncomment and run the cell below:

In [ ]:
# Uncomment to load a saved model
# encoder, decoder, vocab = load_model("best_model.pth", device=cfg.device)
# print("Model loaded successfully!")